
# RCSB PDB/mmCIF Downloader (Range + Multithread + 1k Subfolders)

This notebook downloads **PDB** (or **mmCIF**) files from **RCSB** for a list of PDB IDs stored in a CSV.
It saves files to **Google Drive** under subfolders of size **1,000** (e.g., `0_999`, `1000_1999`, ...),
based on the **absolute row index** (taken from the CSV's **first column**).  
It also supports **resuming** by setting a new index range (e.g., `0–9999` today and `10000–19999` later).

**Key features**
- Range-limited download by absolute index (inclusive).
- Creates deterministic subfolders per 1,000 indices (`0_999`, `1000_1999`, …), independent of the chosen range.
- Skips files that already exist.
- Parallel downloads with retries and polite backoff.
- Writes a log CSV of results for each run.



In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

#%cd /content/drive/MyDrive/LLM/Bioreasoner/testing_place

from pathlib import Path
BASE_DIR = Path("/content/drive/MyDrive/LLM/")
#OUT_DIR  = BASE_DIR / "sft_test_demo"
print(f"Using Google Drive folder as BASE_DIR: {BASE_DIR}")


In [ ]:
%cd /content/drive/MyDrive/

In [ ]:

# (Optional) If needed, install missing packages.
# In Colab, you usually already have `requests`, `pandas`, and `tqdm`.
# Uncomment if you get import errors.
# !pip -q install requests pandas tqdm

import os
import re
import math
import json
import time
from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

import pandas as pd
import requests
from urllib3.util.retry import Retry
from requests.adapters import HTTPAdapter
from tqdm import tqdm


In [ ]:

# =========================
# Configuration
# =========================

# Path to your CSV file. Example: "/content/drive/MyDrive/protein_list.csv"
# The CSV's *first column* must be the absolute index (0..N).
CSV_PATH = "/content/drive/MyDrive/valid_pmcid.csv"

# If you know the PDB ID column name, set it here (e.g., "pdb_id").
# If None, the notebook will try to infer the column automatically.
PDB_ID_COLUMN = "pdb_id"

# Where to save files (Google Drive recommended when on Colab).
BASE_DIR = "/content/drive/MyDrive/PDB"

# Index range to download (inclusive). Example: 0–9999 for the first 10k rows.
START_IDX = 0
END_IDX   = 129999

# Size of each subfolder chunk (1,000 as requested; 0..999, 1000..1999, etc.).
CHUNK_SIZE = 1000

# File format: "pdb" or "cif" (mmCIF). Default is "pdb".
FILE_FORMAT = "pdb"  # or "cif"

# Parallelism
MAX_WORKERS = 8

# HTTP timeouts (connect, read)
TIMEOUT = (5, 30)

# Retry/backoff
RETRY_TOTAL = 5
RETRY_BACKOFF = 2.0
RETRY_STATUS_FORCELIST = [429, 500, 502, 503, 504]


In [ ]:

# =========================
# Helpers
# =========================

def ensure_dir(p: Path) -> None:
    p.mkdir(parents=True, exist_ok=True)

def chunk_folder_for_index(idx: int, chunk_size: int) -> str:
    start = (idx // chunk_size) * chunk_size
    end = start + chunk_size - 1
    return f"{start}_{end}"

def infer_pdb_id_column(df: pd.DataFrame) -> str:
    """
    Try to guess which column contains PDB IDs (4-char alphanumeric).
    We'll check common names first, then scan columns for 4-char tokens.
    """
    candidates_by_name = ["pdb_id", "pdbid", "pdb", "PDB_ID", "PDB", "id"]
    for name in df.columns:
        if name in candidates_by_name:
            # quick validation on a sample
            series = df[name].astype(str).str.strip().str.upper()
            sample = series.head(200)
            valid = sample.str.match(r"^[0-9A-Z]{4}$", na=False).mean()
            if valid > 0.5:  # at least half look like PDB IDs
                return name

    # Fallback: scan every column and pick the first that looks like PDB IDs
    best_col = None
    best_score = -1.0
    for name in df.columns:
        series = df[name].astype(str).str.strip().str.upper()
        sample = series.head(200)
        score = sample.str.match(r"^[0-9A-Z]{4}$", na=False).mean()
        if score > best_score:
            best_score = score
            best_col = name
    if best_col is None or best_score < 0.25:
        raise ValueError("Could not infer PDB ID column; please set PDB_ID_COLUMN explicitly.")
    return best_col

def url_for_pdb(pdb_id: str, fmt: str) -> str:
    pid = pdb_id.strip().upper()
    if fmt == "pdb":
        return f"https://files.rcsb.org/download/{pid}.pdb"
    elif fmt == "cif":
        return f"https://files.rcsb.org/download/{pid}.cif"
    else:
        raise ValueError("FILE_FORMAT must be 'pdb' or 'cif'.")

# Thread-local session factory for safe parallel requests
_thread_local = threading.local()

def get_session() -> requests.Session:
    sess = getattr(_thread_local, "session", None)
    if sess is None:
        sess = requests.Session()
        retries = Retry(
            total=RETRY_TOTAL,
            backoff_factor=RETRY_BACKOFF,
            status_forcelist=RETRY_STATUS_FORCELIST,
            allowed_methods={"GET"},
            raise_on_status=False,
        )
        adapter = HTTPAdapter(max_retries=retries, pool_connections=MAX_WORKERS, pool_maxsize=MAX_WORKERS)
        sess.mount("https://", adapter)
        sess.mount("http://", adapter)
        sess.headers.update({"User-Agent": "RCSB-PDB-Downloader/1.0"})
        _thread_local.session = sess
    return sess

def download_one(idx: int, pdb_id: str, base_dir: Path, fmt: str, chunk_size: int) -> dict:
    """
    Download a single PDB/mmCIF file to the appropriate chunk subfolder.
    Skips if file already exists.
    Returns a dict with fields: index, pdb_id, status, path or error.
    """
    # Resolve output path
    folder = base_dir / chunk_folder_for_index(idx, chunk_size)
    ensure_dir(folder)
    suffix = ".pdb" if fmt == "pdb" else ".cif"
    out_path = folder / f"{pdb_id.upper()}{suffix}"

    # Skip if exists and non-empty
    if out_path.exists() and out_path.stat().st_size > 0:
        return {"index": idx, "pdb_id": pdb_id, "status": "exists", "path": str(out_path)}

    url = url_for_pdb(pdb_id, fmt)
    tmp_path = out_path.with_suffix(out_path.suffix + ".part")

    sess = get_session()
    try:
        with sess.get(url, stream=True, timeout=TIMEOUT) as r:
            if r.status_code == 404:
                return {"index": idx, "pdb_id": pdb_id, "status": "not_found", "error": "404 Not Found"}
            r.raise_for_status()
            with open(tmp_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=1 << 15):
                    if chunk:
                        f.write(chunk)
        os.replace(tmp_path, out_path)
        return {"index": idx, "pdb_id": pdb_id, "status": "downloaded", "path": str(out_path)}
    except Exception as e:
        # Clean partial
        try:
            if tmp_path.exists():
                tmp_path.unlink()
        except Exception:
            pass
        return {"index": idx, "pdb_id": pdb_id, "status": "error", "error": str(e)}


In [ ]:

# =========================
# Run Download
# =========================

BASE_DIR = Path(BASE_DIR)
ensure_dir(BASE_DIR)

# Load CSV, ensure first column is the index (0..N)
df = pd.read_csv(CSV_PATH, index_col=0)
if not pd.api.types.is_integer_dtype(df.index):
    try:
        df.index = df.index.astype(int)
    except Exception:
        raise ValueError("The first column must be an integer index (0..N). Please fix your CSV.")

# Determine PDB ID column
if PDB_ID_COLUMN is None:
    PDB_ID_COLUMN = infer_pdb_id_column(df)
print(f"Using PDB ID column: {PDB_ID_COLUMN!r}")

# Slice by absolute index (inclusive)
start = int(START_IDX)
end   = int(END_IDX)
if start > end:
    raise ValueError("START_IDX must be <= END_IDX")

subset = df.loc[(df.index >= start) & (df.index <= end)]
print(f"Rows to download: {len(subset)} (from index {start} to {end})")

# Prepare tasks
def is_valid_pdb_id(x: str) -> bool:
    if not isinstance(x, str):
        return False
    x = x.strip().upper()
    return bool(re.match(r"^[0-9A-Z]{4}$", x))

records = []
for idx, row in subset.iterrows():
    pid = str(row[PDB_ID_COLUMN]).strip().upper()
    if is_valid_pdb_id(pid):
        records.append((idx, pid))

print(f"Valid PDB IDs in range: {len(records)}")

# Parallel download
results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = [ex.submit(download_one, idx, pid, BASE_DIR, FILE_FORMAT, CHUNK_SIZE) for (idx, pid) in records]
    for fut in tqdm(as_completed(futures), total=len(futures), desc="Downloading"):
        results.append(fut.result())

# Summarize
import collections
ctr = collections.Counter([r["status"] for r in results])
print("\nSummary:", dict(ctr))

# Save log
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
log_path = BASE_DIR / f"rcsb_download_log_{start}_{end}_{ts}.csv"
pd.DataFrame(results).to_csv(log_path, index=False)
print(f"Log saved to: {log_path}")


Using PDB ID column: 'pdb_id'
Rows to download: 110256 (from index 10000 to 129999)
Valid PDB IDs in range: 110256


Downloading:  15%|█▍        | 16471/110256 [19:05<1:39:22, 15.73it/s]

In [ ]:
#Quick Checks

# Count files per chunk folder touched in this range
touched_chunks = sorted({chunk_folder_for_index(i, CHUNK_SIZE) for i in range(START_IDX, END_IDX + 1)})
rows = []
for ch in touched_chunks:
    folder = Path(BASE_DIR) / ch
    n_files = len(list(folder.glob(f"*.{FILE_FORMAT}")))
    rows.append({"chunk": ch, "num_files_in_chunk": n_files})
pd.DataFrame(rows)



## Resume Later
To resume later, just update `START_IDX` and `END_IDX` in the **Configuration** cell (e.g., set to `10000–19999`), then **run the "Run Download" cell again**.  
Already-downloaded files will be **skipped** automatically.


## 3Di token generation

In [ ]:
%cd /content/drive/MyDrive/ #to foldseek binary folder

/content/drive/MyDrive/LLM/Bioreasoner/RCSB_files


In [ ]:
!chmod +x foldseek

In [ ]:
# === Cell: Foldseek config for 3Di export (edit paths as needed) ===

# Range you want to transform (must match your downloader range)
START_IDX = 0
END_IDX   = 129999

# Path to your Foldseek binary
FOLDSEEK_BIN = "/content/drive/MyDrive/foldseek"
FOLDSEEK_VERBOSE = False

# Concurrency for 3Di extraction (per-file parallelism; each call uses --threads 1)
MAX_WORKERS_3DI = 8

# Output CSV naming prefix inside each subfolder (e.g., 3di_chains_10000_10999.csv)
THREEDI_CSV_PREFIX = "3di_chains"

In [ ]:
# === Cell: Utilities for mapping, chunking, and 3Di extraction ===
import os, re, math
from pathlib import Path
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

# Reuse (or redefine if not present)
def chunk_folder_for_index(idx: int, chunk_size: int) -> str:
    start = (idx // chunk_size) * chunk_size
    end = start + chunk_size - 1
    return f"{start}_{end}"

def infer_pdb_id_column(df: pd.DataFrame) -> str:
    # minimal inference (same logic as downloader)
    candidates = ["pdb_id", "pdbid", "pdb", "PDB_ID", "PDB", "id"]
    for name in df.columns:
        if name in candidates:
            s = df[name].astype(str).str.strip().str.upper()
            if s.head(200).str.match(r"^[0-9A-Z]{4}$", na=False).mean() > 0.5:
                return name
    best, best_score = None, -1
    for name in df.columns:
        s = df[name].astype(str).str.strip().str.upper()
        score = s.head(200).str.match(r"^[0-9A-Z]{4}$", na=False).mean()
        if score > best_score:
            best, best_score = name, score
    if best is None or best_score < 0.25:
        raise ValueError("Could not infer PDB ID column; please set PDB_ID_COLUMN explicitly.")
    return best

# Import your helper (contains get_struc_seq using foldseek structureto3didescriptor)
try:
    from foldseek_util import get_struc_seq  # from your uploaded helper
except Exception as e:
    raise RuntimeError(
        "Couldn't import foldseek_util.get_struc_seq. "
        "Please place foldseek_util.py next to this notebook or in the Python path."
    ) from e

In [ ]:
# === Cell: Build the per-file worklist for ONLY the requested index range ===

from pathlib import Path

BASE_DIR = Path(BASE_DIR)
suffix = ".pdb" if FILE_FORMAT == "pdb" else ".cif"

# Load the same CSV used for download to map index -> pdb_id
df_idx = pd.read_csv(CSV_PATH, index_col=0)
if not pd.api.types.is_integer_dtype(df_idx.index):
    df_idx.index = df_idx.index.astype(int)

if PDB_ID_COLUMN is None:
    PDB_ID_COLUMN = infer_pdb_id_column(df_idx)

# Subset
subset = df_idx.loc[(df_idx.index >= START_IDX) & (df_idx.index <= END_IDX)]
print(f"Transforming indices in range: {START_IDX}..{END_IDX} (rows: {len(subset)})")
print(f"Using PDB ID column: {PDB_ID_COLUMN!r}")

work_by_chunk = {} 

for idx, row in subset.iterrows():
    pdb_id = str(row[PDB_ID_COLUMN]).strip().upper()
    if not re.match(r"^[0-9A-Z]{4}$", pdb_id):
        continue
    chunk = chunk_folder_for_index(idx, CHUNK_SIZE)
    fpath = BASE_DIR / chunk / f"{pdb_id}{suffix}"
    if fpath.exists() and fpath.stat().st_size > 0:
        work_by_chunk.setdefault(chunk, []).append((idx, pdb_id, fpath))

# Report
total_files = sum(len(v) for v in work_by_chunk.values())
print(f"Found {total_files} downloaded structures in this range across {len(work_by_chunk)} chunk folder(s).")
for ch, items in sorted(work_by_chunk.items()):
    print(f"  - {ch}: {len(items)} file(s)")

# 6s/1k


Transforming indices in range: 11000..70999 (rows: 60000)
Using PDB ID column: 'pdb_id'
Found 56854 downloaded structures in this range across 60 chunk folder(s).
  - 11000_11999: 947 file(s)
  - 12000_12999: 946 file(s)
  - 13000_13999: 942 file(s)
  - 14000_14999: 948 file(s)
  - 15000_15999: 942 file(s)
  - 16000_16999: 952 file(s)
  - 17000_17999: 942 file(s)
  - 18000_18999: 941 file(s)
  - 19000_19999: 943 file(s)
  - 20000_20999: 953 file(s)
  - 21000_21999: 944 file(s)
  - 22000_22999: 937 file(s)
  - 23000_23999: 944 file(s)
  - 24000_24999: 951 file(s)
  - 25000_25999: 940 file(s)
  - 26000_26999: 932 file(s)
  - 27000_27999: 946 file(s)
  - 28000_28999: 962 file(s)
  - 29000_29999: 954 file(s)
  - 30000_30999: 953 file(s)
  - 31000_31999: 949 file(s)
  - 32000_32999: 947 file(s)
  - 33000_33999: 946 file(s)
  - 34000_34999: 945 file(s)
  - 35000_35999: 938 file(s)
  - 36000_36999: 954 file(s)
  - 37000_37999: 952 file(s)
  - 38000_38999: 934 file(s)
  - 39000_39999: 955 file

In [ ]:
# === Cell: Run Foldseek -> 3Di per file (multithread) WITH PROGRESS BARS, one CSV per subfolder ===

import traceback
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm  # progress bars

def process_one(idx: int, pdb_id: str, path: Path, chunk_name: str):
    """
    Return a list of row dicts (one per chain) or a single error row.
    """
    try:
        # One file per call; threads=1 inside to keep per-call light (we parallelize across files)
        seq_dict = get_struc_seq(
            foldseek=str(FOLDSEEK_BIN),
            path=str(path),
            chains=None,                     # all chains
            process_id=idx,                  # unique-ish temp name for this call
            plddt_mask=False,                # set True if you want masking; requires pLDDT parsing in util
            plddt_threshold=70.0,
            foldseek_verbose=FOLDSEEK_VERBOSE
        )
        rows = []
        for chain_id, (aa_seq, thredi_seq, combined_seq) in seq_dict.items():
            rows.append({
                "index": idx,
                "pdb_id": pdb_id,
                "chain_id": chain_id,      
                "aa_seq": aa_seq,
                "threeDi_seq": thredi_seq,
                "combined_seq": combined_seq,
                "seq_len": len(aa_seq),
                "chunk": chunk_name,
                "path": str(path),
            })
        if not rows:
            return [{"index": idx, "pdb_id": pdb_id, "chain_id": None, "error": "no_chains", "chunk": chunk_name, "path": str(path)}]
        return rows
    except Exception as e:
        return [{
            "index": idx,
            "pdb_id": pdb_id,
            "chain_id": None,
            "error": f"{type(e).__name__}: {e}",
            "trace": traceback.format_exc(limit=1),
            "chunk": chunk_name,
            "path": str(path),
        }]

made = []

_global_total = sum(len(items) for items in work_by_chunk.values())
global_pbar = tqdm(total=_global_total, desc="3Di across chunks", unit="file")

for chunk_name, items in sorted(work_by_chunk.items()):
    out_csv = (BASE_DIR / chunk_name / f"{THREEDI_CSV_PREFIX}_{chunk_name}.csv")
    print(f"\n[Chunk {chunk_name}] → {out_csv.name}  (files: {len(items)})")

    all_rows = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS_3DI) as ex:
        futures = [ex.submit(process_one, idx, pid, fpath, chunk_name) for (idx, pid, fpath) in items]
        for fut in tqdm(as_completed(futures),
                        total=len(futures),
                        desc=f"Chunk {chunk_name}",
                        leave=False,
                        unit="file"):
            res = fut.result()
            all_rows.extend(res)
            global_pbar.update(1)

    df_out = pd.DataFrame(all_rows)

    df_out.sort_values(by=["index", "pdb_id", "chain_id"], inplace=True, ignore_index=True)


    df_out.to_csv(out_csv, index=False)
    made.append((chunk_name, out_csv, len(df_out)))

global_pbar.close()

print("\nDone. CSVs created:")
for ch, p, n in made:
    print(f"  - {ch}: {p}  (rows: {n})")

# 2~3min/1k

3Di across chunks:   0%|          | 0/56854 [00:00<?, ?file/s]


[Chunk 11000_11999] → 3di_chains_11000_11999.csv  (files: 947)


Chunk 11000_11999:   0%|          | 0/947 [00:00<?, ?file/s]


[Chunk 12000_12999] → 3di_chains_12000_12999.csv  (files: 946)


Chunk 12000_12999:   0%|          | 0/946 [00:00<?, ?file/s]

...


[Chunk 69000_69999] → 3di_chains_69000_69999.csv  (files: 946)


Chunk 69000_69999:   0%|          | 0/946 [00:00<?, ?file/s]


[Chunk 70000_70999] → 3di_chains_70000_70999.csv  (files: 952)


Chunk 70000_70999:   0%|          | 0/952 [00:00<?, ?file/s]


Done. CSVs created:
  - 11000_11999: /content/drive/MyDrive/LLM/Bioreasoner/RCSB_files/PDB/11000_11999/3di_chains_11000_11999.csv  (rows: 3548)
  - 12000_12999: /content/drive/MyDrive/LLM/Bioreasoner/RCSB_files/PDB/12000_12999/3di_chains_12000_12999.csv  (rows: 3489)
  - 13000_13999: /content/drive/MyDrive/LLM/Bioreasoner/RCSB_files/PDB/13000_13999/3di_chains_13000_13999.csv  (rows: 3352)
  - 14000_14999: /content/drive/MyDrive/LLM/Bioreasoner/RCSB_files/PDB/14000_14999/3di_chains_14000_14999.csv  (rows: 3629)
  - 15000_15999: /content/drive/MyDrive/LLM/Bioreasoner/RCSB_files/PDB/15000_15999/3di_chains_15000_15999.csv  (rows: 3414)
  - 16000_16999: /content/drive/MyDrive/LLM/Bioreasoner/RCSB_files/PDB/16000_16999/3di_chains_16000_16999.csv  (rows: 3403)
  - 17000_17999: /content/drive/MyDrive/LLM/Bioreasoner/RCSB_files/PDB/17000_17999/3di_chains_17000_17999.csv  (rows: 3521)
  - 18000_18999: /content/drive/MyDrive/LLM/Bioreasoner/RCSB_files/PDB/18000_18999/3di_chains_18000_18999.csv  

In [ ]:

chk = f"{(START_IDX // CHUNK_SIZE) * CHUNK_SIZE}_{((START_IDX // CHUNK_SIZE) * CHUNK_SIZE) + CHUNK_SIZE - 1}"
preview_csv = BASE_DIR / chk / f"{THREEDI_CSV_PREFIX}_{chk}.csv"
if preview_csv.exists():
    display(pd.read_csv(preview_csv).head(10))
else:
    print("No CSV found to preview:", preview_csv)

,index,pdb_id,chain_id,aa_seq,threeDi_seq,combined_seq,seq_len,chunk,path
0,10000,7DM2,A,TVATTPASSPVTLAETGSTLLYPLFNLWGPAFHERYPNVTITAQGT...,DFQADFDLAQEEAEEEAAPLCQVLVVVLQVVVCVVVVSYHYYYYYA...,TdVfAqTaTdPfAdSlSaPqVeTeLaAeEeTeGaSaTpLlLcYqPv...,334,10000_10999,/content/drive/MyDrive/LLM/Bioreasoner/RCSB_fi...
1,10000,7DM2,H,EVQLVESGGGLVKPGGSLRLSCAASGFTFSSHRMHWVRQAPGKGLE...,DKAKAKDFAAEAAQQAKTKIKIAIDDDQLLFWWKFKWWAAVPGDID...,EdVkQaLkVaEkSdGfGaGaLeVaKaPqGqGaSkLtRkLiSkCiAa...,217,10000_10999,/content/drive/MyDrive/LLM/Bioreasoner/RCSB_fi...
2,10000,7DM2,L,SVLTQTPSASGTPGQRVTISCSGSRSNIGSNYVYWFQQFPGAAPQL...,DQKAWDQEDEDEAQAKDKIKIAHACQWLQPWFKWKWWDAPPDDIDT...,SdVqLkTaQwTdPqSeAdSeGdTePaGqQaRkVdTkIiSkCiSaGh...,211,10000_10999,/content/drive/MyDrive/LLM/Bioreasoner/RCSB_fi...
3,10001,7GNE,A,SGFRKMAFPSGKVEGCMVQVTCGTTTLNGLWLDDVVYCPRHVICTS...,DADDQDADDCVLQLLQWKWKAADPAIAIWGAAFQKTKFFLCNLPDP...,SdGaFdRdKqMdAaFdPdScGvKlVqElGlCqMwVkQwVkTaCaGd...,305,10000_10999,/content/drive/MyDrive/LLM/Bioreasoner/RCSB_fi...
4,10001,7GNE,B,SGFRKMAFPSGKVEGCMVQVTCGTTTLNGLWLDDVVYCPRHVICTS...,DADDQDAADCVLQLLQWKWKAADPDIAIWGAAFQKTKFFLCNLDDL...,SdGaFdRdKqMdAaFaPdScGvKlVqElGlCqMwVkQwVkTaCaGd...,305,10000_10999,/content/drive/MyDrive/LLM/Bioreasoner/RCSB_fi...
5,10002,7JPE,A,ASSQAWQPGVAMPNLYKMQRMLLEKCDLQNYGDSATLPKGIMMNVA...,DDPCLLAQFDWDDPVLQPDQFAADDFDFPLAPDFFDADPPDDLLLL...,AdSdSpQcAlWlQaPqGfVdAwMdPdNpLvYlKqMpQdRqMfLaLa...,299,10000_10999,/content/drive/MyDrive/LLM/Bioreasoner/RCSB_fi...
6,10002,7JPE,B,AFAVDAAKAYKDYLASGGQPITNCVKMLCTHTGTGQAITVTPEANM...,DDDPPVLVVVVVCVVVVHQADDPFAWDDDPQCAPQAFKALDDRHHP...,AdFdAdVpDpAvAlKvAvYvKvDvYcLvAvSvGvGhQqPaIdTdNp...,115,10000_10999,/content/drive/MyDrive/LLM/Bioreasoner/RCSB_fi...
7,10003,7LP4,A,KVTQSFLPPGWEMRIAPNGRPFFIDHNTKTTTWEDPRLKFPVHMRSK,DPDDDDDDPQWDWDADPVGAIWIAGNPVRDIDGDDPCVVPDDVVVPD,KdVpTdQdSdFdLdPdPpGqWwEdMwRdIaAdPpNvGgRaPiFwFi...,47,10000_10999,/content/drive/MyDrive/LLM/Bioreasoner/RCSB_fi...
8,10004,7MP1,A,NHVIFKKISRDKSVTIYLGKRDYIDHVERVEPVDGVVLVDPELVKG...,DQAWAWDAFLLRFKIKIWSDLEWEDALVFTRWTKIKIQGPCVVQPQ...,NdHqVaIwFaKwKdIaSfRlDlKrSfVkTiIkYiLwGsKdRlDeYw...,370,10000_10999,/content/drive/MyDrive/LLM/Bioreasoner/RCSB_fi...
9,10004,7MP1,B,NHVIFKKISRDKSVTIYLGKRDYIDHVERVEPVDGVVLVDPELVKG...,DQAWAKDAFLLRQKIKIWSDLAWEDALVFTRWTKIKIQGPCVVQPQ...,NdHqVaIwFaKkKdIaSfRlDlKrSqVkTiIkYiLwGsKdRlDaYw...,361,10000_10999,/content/drive/MyDrive/LLM/Bioreasoner/RCSB_fi...


### Merging Everything

In [ ]:
# === Merge all 3di_chains_chaintag_*_* per-chunk CSVs, audit missing/short ones ===
from pathlib import Path
import re, os
import pandas as pd
from tqdm.notebook import tqdm

BASE_DIR = Path(BASE_DIR)  # reuse your existing BASE_DIR variable
MERGED_OUT = BASE_DIR / "3di_chains_chaintag_all.csv"
AUDIT_OUT  = BASE_DIR / "3di_chaintag_merge_audit.csv"

# If True, exclude chunks with < MIN_ROWS_PER_CHUNK from the merged file.
EXCLUDE_TOO_SMALL = False
MIN_ROWS_PER_CHUNK = 900

# Find chunk folders like "0_999", "1000_1999", ...
chunk_pat = re.compile(r"^\d+_\d+$")
chunk_dirs = [p for p in BASE_DIR.iterdir() if p.is_dir() and chunk_pat.match(p.name)]

# Sort by numeric start,end for stable order
def _chunk_key(p: Path):
    s, e = p.name.split("_", 1)
    return (int(s), int(e))
chunk_dirs = sorted(chunk_dirs, key=_chunk_key)

def _count_rows_quick(csv_path: Path) -> int:
    # Count data lines quickly (total lines - 1 for header). Returns 0 if file empty/missing header.
    try:
        with open(csv_path, "rb") as fh:
            n_lines = sum(1 for _ in fh)
        return max(n_lines - 1, 0)
    except Exception:
        return 0

audit_rows = []
present_files = []

# Scan & audit
for d in tqdm(chunk_dirs, desc="Scan chunks", leave=True):
    chunk = d.name
    csv_path = d / f"3di_chains_chaintag_{chunk}.csv"
    if not csv_path.exists():
        audit_rows.append({"chunk": chunk, "csv_path": str(csv_path), "exists": False,
                           "rows": 0, "status": "missing"})
        continue
    rows = _count_rows_quick(csv_path)
    status = "ok" if rows >= MIN_ROWS_PER_CHUNK else "too_small"
    audit_rows.append({"chunk": chunk, "csv_path": str(csv_path), "exists": True,
                       "rows": rows, "status": status})
    present_files.append((csv_path, rows, status))

# Write audit table
audit_df = pd.DataFrame(audit_rows).sort_values(
    by=["exists","status","chunk"], ascending=[False, True, True]
)
audit_df.to_csv(AUDIT_OUT, index=False)
print(f"[audit] wrote {AUDIT_OUT}")

# Report summary
n_total_chunks = len(chunk_dirs)
n_missing = sum(1 for r in audit_rows if r["status"] == "missing")
n_toosmall = sum(1 for r in audit_rows if r["status"] == "too_small")
n_present = sum(1 for r in audit_rows if r["exists"])
print(f"[summary] chunks: total={n_total_chunks}, present={n_present}, missing={n_missing}, <{MIN_ROWS_PER_CHUNK} rows={n_toosmall}")

# Merge (optionally exclude too-small)
if MERGED_OUT.exists():
    MERGED_OUT.unlink()  # start fresh

written_rows = 0
files_for_merge = [(p, r, s) for (p, r, s) in present_files if (s != "too_small" or not EXCLUDE_TOO_SMALL)]

with tqdm(total=len(files_for_merge), desc="Merging", leave=True) as pbar_merge:
    first = True
    for csv_path, rows, status in files_for_merge:
        # chunked read to keep memory reasonable
        try:
            for chunk_df in pd.read_csv(csv_path, chunksize=200_000):
                chunk_df.to_csv(MERGED_OUT, mode="a", index=False, header=first)
                first = False
                written_rows += len(chunk_df)
        except Exception as e:
            print(f"[warn] failed reading {csv_path}: {e}")
        pbar_merge.update(1)

print(f"[done] merged rows written: {written_rows}")
print(f"[out] merged file: {MERGED_OUT}")
print(f"[out] audit file:  {AUDIT_OUT}")

# Show a quick summary table inline
display(audit_df.head(20))

### Filtering repeated rows

In [ ]:
# === Cell 1: per-protein summary from merged CSV ===
from pathlib import Path
import pandas as pd
import hashlib

# Prefer an "all" merged if present, else the original merged
src_all    = Path(BASE_DIR) / "3di_chains_chaintag_all.csv"
src_merged = Path(BASE_DIR) / "3di_chains_chaintag_merged.csv"
SRC = src_all if src_all.exists() else src_merged
assert SRC.exists(), f"Missing merged CSV at {src_all} or {src_merged}"

# ----- config -----
SEQUENCE_FIELD  = "aa_seq"                 # or "threeDi_seq" / "combined_seq"
INCLUDE_CLASSES = {"prot","other","DNA","RNA"}         # include D-peptides if labeled 'other'
CHAIN_ORDER     = "chain_id"               # "chain_id" or "as_is"
NORMALIZE       = True                     # uppercase & strip spaces
OUT1 = Path(BASE_DIR) / "protein_summary_by_pdb_aa_seq.csv"
# ------------------

# robust load
try:
    df = pd.read_csv(SRC, dtype=str)
except Exception as e:
    print("[warn] strict read failed, tolerant parser:", e)
    df = pd.read_csv(SRC, dtype=str, engine="python", on_bad_lines="skip")

need = {"pdb_id","chain_id","polymer_class",SEQUENCE_FIELD}
missing = need - set(df.columns)
if missing:
    raise ValueError(f"Merged CSV missing columns: {sorted(missing)}")

df["polymer_class"] = df["polymer_class"].fillna("other").str.lower()
df = df[df["polymer_class"].isin({c.lower() for c in INCLUDE_CLASSES})].copy()

df[SEQUENCE_FIELD] = df[SEQUENCE_FIELD].fillna("").astype(str)
df = df[df[SEQUENCE_FIELD].str.len() > 0].copy()

def normalize_seq(s: str) -> str:
    s2 = s.strip()
    if NORMALIZE:
        s2 = s2.upper().replace(" ", "")
        # uncomment to force only letters:
        # s2 = "".join(ch for ch in s2 if ch.isalpha())
    return s2

df["seq_norm"] = df[SEQUENCE_FIELD].map(normalize_seq)
df["_sort_key"] = df["chain_id"].astype(str) if CHAIN_ORDER == "chain_id" else df.index

def concat_group(g: pd.DataFrame) -> dict:
    g2 = g.sort_values("_sort_key")
    seqs = g2["seq_norm"].tolist()
    conc = "".join(seqs)
    return {
        "merged_sequence": conc,
        "concat_md5": hashlib.md5(conc.encode("utf-8")).hexdigest(),
        "n_chains": len(g2),
        "total_length": sum(len(s) for s in seqs),
        "chain_ids": ",".join(g2["chain_id"].astype(str).tolist()),
    }

per_protein = (
    df.groupby("pdb_id", sort=False)
      .apply(concat_group)
      .apply(pd.Series)
      .reset_index()
)

per_protein.to_csv(OUT1, index=False)
print(f"[ok] wrote per-protein summary -> {OUT1}")
display(per_protein.head(10))

/tmp/ipython-input-1433687841.py:63: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(concat_group)


[ok] wrote per-protein summary -> /content/drive/MyDrive/LLM/Bioreasoner/RCSB_files/protein_summary_by_pdb_aa_seq.csv


,pdb_id,merged_sequence,concat_md5,n_chains,total_length,chain_ids
0,100D,XXXXXXXXXXXXXXXXXXXX,1cb17f857d5b4bab8268d22f376ce61c,2,20,"A,B"
1,1OEB,PLGSVRWARALYDFEALEEDELGFRSGEVVEVLDSSNPSWWTGRLH...,53ba63d985737c19a9b6467108541bcc,4,138,"A,B,C,D"
2,2AX8,IFLNVLEAIEPGVVCAGHDNNQPDSFAALLSSLNELGERQLVHVVK...,2ab179c363d7d54c4586b43853fcb69e,1,238,A
3,2CZJ,APVLENRRARHDYEILETYEAGIALKGTEVKSLRAGKVDFTGSFAR...,702406602c0bda265540eb32e177199a,8,722,"A,B,C,D,E,F,G,H"
4,2IC2,GSTYPPTPPNVTRLSVMLRWMVPRNDGLPIVIFKVQYRMVGNWQTT...,b24ec3a66a9359844ba13d0fe2a7032c,2,218,"A,B"
5,2JG9,QPRPAFSAIRRNPPMGGNVVIFDTVITNQEEPYQNHSGRFVCTVPG...,6d2cd03769592b0afb30cbfe361c4172,6,792,"A,B,C,D,E,F"
6,2KHY,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX,625d4570022c770c2565ca7d1bd5cb3f,1,38,A
7,2MJO,MTRGTTDNLIPVYASILAAVVVGLVAYIAFKRWNSSKQNKQMTRGT...,689061dc180d6c49d4d9cd03a0264d0a,2,82,"A,B"
8,2QVJ,SVTVKRIIDNTVIVPKLPANEDPVEYPADYFRKSKEIPLYINTTKS...,102696752961f3b27f09e0be76a80208,5,2105,"A,B,C,D,E"
9,2WQS,AMVNKNKEGLNIDGKEVLAGSTNYYELTWDLDQYKGDKSSKEAIQN...,228dea5c70722ea3efb81594ecab8bf2,1,333,A


In [ ]:
len(per_protein)

111629

In [ ]:
# === Cell 2: merge per-protein summary with valid_pmcid.csv ===
from pathlib import Path
import pandas as pd

IN1  = Path(BASE_DIR) / "protein_summary_by_pdb_aa_seq.csv"
assert IN1.exists(), f"Missing input: {IN1}"

# try common locations for valid_pmcid.csv
VALID_PMCID_PATHS = [
    Path(BASE_DIR) / "valid_pmcid.csv",
    Path("/mnt/data/valid_pmcid.csv"),
]
valid_pmcid_csv = next((p for p in VALID_PMCID_PATHS if p.exists()), None)
assert valid_pmcid_csv is not None, f"valid_pmcid.csv not found in: {VALID_PMCID_PATHS}"

OUT2 = Path(BASE_DIR) / "protein_summary_with_paper.csv"

pp = pd.read_csv(IN1, dtype=str)
meta = pd.read_csv(valid_pmcid_csv, dtype=str)
meta.columns = [c.strip().lower() for c in meta.columns]

# find a pdb id column in metadata
pdb_key_candidates = ["pdb_id","pdb","pdbid","structure_id","structureid"]
pdb_col = next((c for c in pdb_key_candidates if c in meta.columns), None)
if pdb_col is None:
    raise ValueError(f"No PDB id column found in {valid_pmcid_csv} (tried {pdb_key_candidates})")

meta["pdb_id"] = meta[pdb_col].astype(str).str.upper()
pp["pdb_id"]   = pp["pdb_id"].astype(str).str.upper()

want_cols = ["title","doi","pmid","journal","year","release_date","authors","pmcid"]
for c in want_cols:
    if c not in meta.columns:
        meta[c] = pd.NA

meta_slim = meta[["pdb_id"] + want_cols].drop_duplicates("pdb_id")

merged = pp.merge(meta_slim, on="pdb_id", how="left")
merged.to_csv(OUT2, index=False)
print(f"[ok] wrote merged summary -> {OUT2}")
display(merged.head(10))


[ok] wrote merged summary -> /content/drive/MyDrive/LLM/Bioreasoner/RCSB_files/protein_summary_with_paper.csv


,pdb_id,merged_sequence,concat_md5,n_chains,total_length,chain_ids,title,doi,pmid,journal,year,release_date,authors,pmcid
0,100D,XXXXXXXXXXXXXXXXXXXX,1cb17f857d5b4bab8268d22f376ce61c,2,20,"A,B",Crystal structure of the highly distorted chim...,10.1093/nar/22.24.5466,7816639,Nucleic Acids Res.,1994.0,1995-03-31T00:00:00Z,"Ban, C., Ramakrishnan, B., Sundaralingam, M.",PMC332097
1,1OEB,PLGSVRWARALYDFEALEEDELGFRSGEVVEVLDSSNPSWWTGRLH...,53ba63d985737c19a9b6467108541bcc,4,138,"A,B,C,D",Structural Basis for SH3 Domain-Mediated High-...,10.1093/EMBOJ/CDG258,12773374,Embo J.,2003.0,2003-04-02T00:00:00Z,"Harkiolaki, M., Lewitzky, M., Gilbert, R.J.C.,...",PMC156755
2,2AX8,IFLNVLEAIEPGVVCAGHDNNQPDSFAALLSSLNELGERQLVHVVK...,2ab179c363d7d54c4586b43853fcb69e,1,238,A,Structural Basis for Accommodation of Nonstero...,10.1074/jbc.M507464200,16129672,J.Biol.Chem.,2005.0,2005-09-20T00:00:00Z,"Bohl, C.E., Miller, D.D., Chen, J., Bell, C.E....",PMC2072880
3,2CZJ,APVLENRRARHDYEILETYEAGIALKGTEVKSLRAGKVDFTGSFAR...,702406602c0bda265540eb32e177199a,8,722,"A,B,C,D,E,F,G,H",Structural basis for functional mimicry of lon...,10.1073/pnas.0700402104,17488812,Proc.Natl.Acad.Sci.Usa,2007.0,2006-10-31T00:00:00Z,"Bessho, Y., Shibata, R., Sekine, S., Murayama,...",PMC1895943
4,2IC2,GSTYPPTPPNVTRLSVMLRWMVPRNDGLPIVIFKVQYRMVGNWQTT...,b24ec3a66a9359844ba13d0fe2a7032c,2,218,"A,B",Structure of a heparin-dependent complex of He...,10.1073/pnas.0606738103,17077139,Proc.Natl.Acad.Sci.Usa,2006.0,2006-10-24T00:00:00Z,"McLellan, J.S., Leahy, D.J.",PMC1859911
5,2JG9,QPRPAFSAIRRNPPMGGNVVIFDTVITNQEEPYQNHSGRFVCTVPG...,6d2cd03769592b0afb30cbfe361c4172,6,792,"A,B,C,D,E,F",C1Q Binds Phosphatidylserine and Likely Acts a...,10.4049/jimmunol.180.4.2329,18250442,J.Immunol.,2008.0,2008-02-19T00:00:00Z,"Paidassi, H., Tacnet-Delorme, P., Garlatti, V....",PMC2632962
6,2KHY,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX,625d4570022c770c2565ca7d1bd5cb3f,1,38,A,NMR structure and dynamics of the Specifier Lo...,10.1093/nar/gkq020,20110252,Nucleic Acids Res.,2010.0,2010-02-16T00:00:00Z,"Wang, J., Henkin, T.M.",PMC2879506
7,2MJO,MTRGTTDNLIPVYASILAAVVVGLVAYIAFKRWNSSKQNKQMTRGT...,689061dc180d6c49d4d9cd03a0264d0a,2,82,"A,B",Structural Basis of p75 Transmembrane Domain D...,10.1074/jbc.M116.723585,27056327,J. Biol. Chem.,2016.0,2015-01-28T00:00:00Z,"Nadezhdin, K., Arseniev, A., Goncharuk, S., Mi...",PMC4933281
8,2QVJ,SVTVKRIIDNTVIVPKLPANEDPVEYPADYFRKSKEIPLYINTTKS...,102696752961f3b27f09e0be76a80208,5,2105,"A,B,C,D,E",Role of intermolecular interactions of vesicul...,10.1128/JVI.00935-07,18003727,J.Virol.,2008.0,2008-01-08T00:00:00Z,"Luo, M., Green, T.J., Zhang, X., Tsao, J., Qiu...",PMC2224587
9,2WQS,AMVNKNKEGLNIDGKEVLAGSTNYYELTWDLDQYKGDKSSKEAIQN...,228dea5c70722ea3efb81594ecab8bf2,1,333,A,Two Intramolecular Isopeptide Bonds are Identi...,10.1016/J.JMB.2010.01.065,20138058,J.Mol.Biol.,2010.0,2010-02-16T00:00:00Z,"Forsgren, N., Persson, K.",PMC2849992


In [ ]:
len(merged)

111629

In [ ]:
# === Cell 3: filter repeated rows by identical sequence + paper metadata ===
from pathlib import Path
import pandas as pd

IN2  = Path(BASE_DIR) / "protein_summary_with_paper.csv"
assert IN2.exists(), f"Missing input: {IN2}"

OUT3 = Path(BASE_DIR) / "repeated_proteins_by_metadata.csv"

df = pd.read_csv(IN2, dtype=str)

# normalize comparison fields (fillna with empty strings, strip)
for col in ["merged_sequence","doi","title","authors","pmcid"]:
    if col not in df.columns:
        df[col] = ""
    df[col] = df[col].fillna("").astype(str).str.strip()

# group key for repeats
key_cols = ["merged_sequence","doi","title","authors","pmcid"]
grp_sz = df.groupby(key_cols)["pdb_id"].transform("size")
repeats = df.loc[grp_sz > 1].copy()

# sort stably
repeats = repeats.sort_values(key_cols + ["pdb_id"])
repeats.to_csv(OUT3, index=False)
print(f"[ok] wrote repeated-by-metadata rows -> {OUT3} (rows: {len(repeats)})")
display(repeats.head(20))


[ok] wrote repeated-by-metadata rows -> /content/drive/MyDrive/LLM/Bioreasoner/RCSB_files/repeated_proteins_by_metadata.csv (rows: 21913)


,pdb_id,merged_sequence,concat_md5,n_chains,total_length,chain_ids,title,doi,pmid,journal,year,release_date,authors,pmcid
29256,6XOS,AAACERALQYKLGDKIHGFTVNQVTSVPELFLTAVKLTHDDTGARY...,968769d61763debeaca404066402f49d,1,966,A,Structural basis for the mechanisms of human p...,10.1038/s41467-022-29322-4,35383169,Nat Commun,2022.0,2021-07-07T00:00:00Z,"Liang, W.G., Zhao, M., Tang, W.",PMC8983764
48239,6XOT,AAACERALQYKLGDKIHGFTVNQVTSVPELFLTAVKLTHDDTGARY...,968769d61763debeaca404066402f49d,1,966,A,Structural basis for the mechanisms of human p...,10.1038/s41467-022-29322-4,35383169,Nat Commun,2022.0,2021-07-07T00:00:00Z,"Liang, W.G., Zhao, M., Tang, W.",PMC8983764
54343,3ATU,AAAIGIDLGTTYSCVGVFQHGKVEIIANDQGNRTTPSYVAFTDTER...,f69f94b5418e1a09f543250fede4c572,1,379,A,Biochemical and structural studies on the high...,10.1002/pro.663,21608060,Protein Sci.,2011.0,2011-12-28T00:00:00Z,"Arakawa, A., Handa, N., Shirouzu, M., Yokoyama...",PMC3189522
1685,3AY9,AAAIGIDLGTTYSCVGVFQHGKVEIIANDQGNRTTPSYVAFTDTER...,f69f94b5418e1a09f543250fede4c572,1,379,A,Biochemical and structural studies on the high...,10.1002/pro.663,21608060,Protein Sci.,2011.0,2011-12-28T00:00:00Z,"Arakawa, A., Handa, N., Shirouzu, M., Yokoyama...",PMC3189522
1463,7E84,AAGVAAWLPFARAAAIGWMPVASGPMPAPPRQERDALIVLNVSGTR...,5f3692cfc07f1d4d4042e7d90fa3a6f4,8,2548,"A,B,D,E,G,H,J,L",Structural basis of gating modulation of Kv4 c...,10.1038/s41586-021-03935-z,34552243,Nature,2021.0,2021-10-13T00:00:00Z,"Kise, Y., Nureki, O.",PMC8566240
71954,7F3F,AAGVAAWLPFARAAAIGWMPVASGPMPAPPRQERDALIVLNVSGTR...,5f3692cfc07f1d4d4042e7d90fa3a6f4,8,2548,"A,B,D,E,G,H,J,L",Structural basis of gating modulation of Kv4 c...,10.1038/s41586-021-03935-z,34552243,Nature,2021.0,2021-10-13T00:00:00Z,"Kise, Y., Nureki, O.",PMC8566240
27244,7ENZ,AAHGSNPEQLKHCNGILKELLSKKHAAYAWPFYKPVDASALGLHDY...,60c2967edcf76b73242618a46369898b,1,115,A,"Structural investigation of a pyrano-1,3-oxazi...",10.1107/S2053230X22001066,35234137,"Acta Crystallogr.,Sect.F",2022.0,2022-03-09T00:00:00Z,"Padmanabhan, B., Arole, A., Deshmukh, P., Asho...",PMC8900734
9376,7EO5,AAHGSNPEQLKHCNGILKELLSKKHAAYAWPFYKPVDASALGLHDY...,60c2967edcf76b73242618a46369898b,1,115,A,"Structural investigation of a pyrano-1,3-oxazi...",10.1107/S2053230X22001066,35234137,"Acta Crystallogr.,Sect.F",2022.0,2022-03-09T00:00:00Z,"Padmanabhan, B., Arole, A., Deshmukh, P., Asho...",PMC8900734
59378,3I58,AAHIGLRALADLATPMAVRVAATLRVADHIAAGHRTAAEIASAAGA...,65986514710f3c165afef47689a7e961,2,656,"A,B",Molecular basis of substrate promiscuity for t...,10.1021/bi901257q,19702337,Biochemistry,2009.0,2009-09-01T00:00:00Z,"Cooke, H.A., Bruner, S.D.",PMC4500170
97522,3I64,AAHIGLRALADLATPMAVRVAATLRVADHIAAGHRTAAEIASAAGA...,65986514710f3c165afef47689a7e961,2,656,"A,B",Molecular basis of substrate promiscuity for t...,10.1021/bi901257q,19702337,Biochemistry,2009.0,2009-09-01T00:00:00Z,"Cooke, H.A., Bruner, S.D.",PMC4500170


In [ ]:
len(repeats)

21913

In [ ]:
# === Cell 4: FASTA-verify repeated rows and keep those with identical FASTA sequences (within metadata groups) ===
from pathlib import Path
import io, re, time, requests, hashlib
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

IN3  = Path(BASE_DIR) / "repeated_proteins_by_metadata.csv"
assert IN3.exists(), f"Missing input: {IN3}"

OUT_ALL  = Path(BASE_DIR) / "repeated_proteins_with_fasta.csv"
OUT_KEEP = Path(BASE_DIR) / "repeated_proteins_fasta_verified.csv"
FASTA_DIR = Path(BASE_DIR) / "FASTA_repeat_verify"
FASTA_DIR.mkdir(parents=True, exist_ok=True)

# HTTP settings
TIMEOUT = (5, 30)         # (connect, read)
RETRY_TRIES = 4
RETRY_BACKOFF = 1.5
MAX_WORKERS = 12          # parallel fetches

# Metadata grouping columns (must match Cell 3 logic)
META_KEYS = ["merged_sequence", "doi", "title", "authors", "pmcid"]

# ----------------- helpers -----------------
def http_get_with_retries(url, timeout=TIMEOUT, tries=RETRY_TRIES, backoff=RETRY_BACKOFF):
    last_exc = None
    for i in range(tries):
        try:
            r = requests.get(url, timeout=timeout)
            if r.status_code == 200 and r.text.strip():
                return r.text
        except Exception as e:
            last_exc = e
        time.sleep(backoff**i)
    if last_exc:
        raise last_exc
    raise RuntimeError(f"Failed to fetch after {tries} attempts: {url}")

def fetch_fasta(pdb_id: str) -> tuple[str, str]:
    """Return (fasta_text, source) or raise."""
    pdb_id_up = pdb_id.upper().strip()
    urls = [
        f"https://www.rcsb.org/fasta/entry/{pdb_id_up}",
        f"https://www.ebi.ac.uk/pdbe/entry-files/download/fasta/{pdb_id_up.lower()}.fasta",
    ]
    for u in urls:
        try:
            txt = http_get_with_retries(u)
            return txt, ("RCSB" if "rcsb.org" in u else "PDBe")
        except Exception:
            continue
    raise RuntimeError(f"Could not fetch FASTA for {pdb_id_up}")

def parse_fasta_records(text: str):
    """Return list of {'header': str, 'seq': str} for all records in the FASTA text."""
    rows, header, parts = [], None, []
    for line in io.StringIO(text):
        line = line.rstrip("\n")
        if line.startswith(">"):
            if header is not None:
                rows.append({"header": header, "seq": "".join(parts)})
            header, parts = line[1:].strip(), []
        else:
            parts.append(line.strip())
    if header is not None:
        rows.append({"header": header, "seq": "".join(parts)})
    return rows

_CHAIN_RE = re.compile(r"[Cc]hain[s]?\s+([A-Za-z0-9,; ]+)")
def extract_chain_ids_from_header(h: str):
    # Try "Chain X" / "Chains A,B"
    m = _CHAIN_RE.search(h)
    if m:
        return re.findall(r"[A-Za-z0-9]", m.group(1))
    # Try pipe-delimited tokens like |A|
    parts = [t for t in h.split("|") if t]
    ids = [t for t in parts if re.fullmatch(r"[A-Za-z0-9]", t)]
    if ids:
        return ids
    # Fallback: loose tokenization if header mentions the PDB ID (avoid over-matching)
    ids = re.findall(r"(?:^|[\s|:_-])([A-Za-z0-9])(?:\s|$)", h)
    return sorted(set(ids)) if ids else []

def normalize_aa_sequence(s: str) -> str:
    s = s.upper()
    # keep letters only; keeps 'X' (unknown/mod)
    return "".join(ch for ch in s if "A" <= ch <= "Z")

def concat_entry_sequence(fasta_rows):
    """
    Deterministic per-entry concatenation:
    - normalize each chain sequence
    - sort chains by parsed chain id (fallback "~" to sort last if none)
    - concatenate
    """
    chains = []
    for r in fasta_rows:
        seq = normalize_aa_sequence(r["seq"])
        if not seq:
            continue
        ids = extract_chain_ids_from_header(r["header"])
        chain_id = ids[0] if ids else "~"
        chains.append((chain_id, seq))
    if not chains:
        return "", 0
    chains.sort(key=lambda x: x[0])
    cat = "".join(seq for _, seq in chains)
    return cat, len(chains)

def process_one_pdb(pdb_id: str):
    """Fetch, save, parse, and summarize one PDB's FASTA."""
    pdb_up = pdb_id.upper()
    out_path = FASTA_DIR / f"{pdb_up}.fasta"
    try:
        fasta_text, source = fetch_fasta(pdb_up)
        # save
        with open(out_path, "w") as fh:
            fh.write(fasta_text)
        # parse & concat
        recs = parse_fasta_records(fasta_text)
        concat_seq, n_used = concat_entry_sequence(recs)
        md5 = hashlib.md5(concat_seq.encode()).hexdigest() if concat_seq else ""
        return {
            "pdb_id": pdb_up,
            "fasta_source": source,
            "fasta_path": str(out_path),
            "n_fasta_records": len(recs),
            "n_chains_used": n_used,
            "fasta_concat_length": len(concat_seq),
            "fasta_concat_md5": md5,
            "fasta_concat_sequence": concat_seq,
            "fasta_ok": True,
            "error": "",
        }
    except Exception as e:
        return {
            "pdb_id": pdb_up,
            "fasta_source": "",
            "fasta_path": "",
            "n_fasta_records": 0,
            "n_chains_used": 0,
            "fasta_concat_length": 0,
            "fasta_concat_md5": "",
            "fasta_concat_sequence": "",
            "fasta_ok": False,
            "error": str(e),
        }

# ----------------- main -----------------
rep = pd.read_csv(IN3, dtype=str)

# Normalize comparison fields (same as Cell 3)
for col in META_KEYS + ["pdb_id"]:
    if col not in rep.columns:
        rep[col] = ""
    rep[col] = rep[col].fillna("").astype(str).str.strip()
rep["pdb_id"] = rep["pdb_id"].str.upper()

# Unique PDB IDs to fetch
pdbs = sorted(rep["pdb_id"].dropna().unique().tolist())

# Fetch in parallel with a progress bar
results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futs = {ex.submit(process_one_pdb, pid): pid for pid in pdbs}
    for fut in tqdm(as_completed(futs), total=len(futs), desc="FASTA fetch+parse", leave=True):
        results.append(fut.result())

fasta_df = pd.DataFrame(results)

# Merge FASTA info back to all repeated rows (keep all existing columns)
rep_w_fa = rep.merge(fasta_df, on="pdb_id", how="left")

# Within each metadata group, keep only rows whose FASTA concat matches at least one other row
group_key = rep_w_fa[META_KEYS].apply(lambda r: tuple(r.values.tolist()), axis=1)
rep_w_fa["__group_key__"] = group_key

# mark duplicates-by-FASTA inside each metadata group
rep_w_fa["__md5_group_size__"] = (
    rep_w_fa.groupby(["__group_key__", "fasta_concat_md5"])["pdb_id"].transform("size")
)

# A valid duplicate must have a non-empty md5 and group size >= 2
mask_keep = (rep_w_fa["fasta_concat_md5"].astype(str) != "") & (rep_w_fa["__md5_group_size__"] >= 2)
rep_keep = rep_w_fa.loc[mask_keep].copy()

# (Optional) diagnostic: whether merged_sequence equals FASTA concat (after normalization)
def _norm(s: str) -> str:
    s = (s or "").upper().replace(" ", "")
    return "".join(ch for ch in s if "A" <= ch <= "Z")
if "merged_sequence" in rep_keep.columns:
    rep_keep["merged_equals_fasta"] = rep_keep.apply(
        lambda r: _norm(r["merged_sequence"]) == _norm(r.get("fasta_concat_sequence", "")), axis=1
    )

# Clean helper columns
rep_w_fa = rep_w_fa.drop(columns=["__group_key__", "__md5_group_size__"], errors="ignore")
rep_keep = rep_keep.drop(columns=["__group_key__", "__md5_group_size__"], errors="ignore")

# Save both: full with FASTA, and filtered FASTA-verified duplicates
rep_w_fa.to_csv(OUT_ALL, index=False)
rep_keep.to_csv(OUT_KEEP, index=False)

print(f"[ok] wrote all repeated rows + FASTA -> {OUT_ALL} (rows: {len(rep_w_fa)})")
print(f"[ok] wrote FASTA-verified duplicates -> {OUT_KEEP} (rows: {len(rep_keep)})")

# Quick peeks
display(rep_w_fa.head(10))
if len(rep_keep):
    print("\n[peek] FASTA-verified duplicate groups (first 20 rows):")
    display(rep_keep.head(20))
else:
    print("\n[info] No FASTA-identical duplicates found within metadata groups.")



FASTA fetch+parse:   0%|          | 0/21913 [00:00<?, ?it/s]

[ok] wrote all repeated rows + FASTA -> /content/drive/MyDrive/LLM/Bioreasoner/RCSB_files/repeated_proteins_with_fasta.csv (rows: 21913)
[ok] wrote FASTA-verified duplicates -> /content/drive/MyDrive/LLM/Bioreasoner/RCSB_files/repeated_proteins_fasta_verified.csv (rows: 19655)


,pdb_id,merged_sequence,concat_md5,n_chains,total_length,chain_ids,title,doi,pmid,journal,...,pmcid,fasta_source,fasta_path,n_fasta_records,n_chains_used,fasta_concat_length,fasta_concat_md5,fasta_concat_sequence,fasta_ok,error
0,6XOS,AAACERALQYKLGDKIHGFTVNQVTSVPELFLTAVKLTHDDTGARY...,968769d61763debeaca404066402f49d,1,966,A,Structural basis for the mechanisms of human p...,10.1038/s41467-022-29322-4,35383169,Nat Commun,...,PMC8983764,RCSB,/content/drive/MyDrive/LLM/Bioreasoner/RCSB_fi...,1,1,1014,d8a0f9bb3577eef783246038c5f4832a,MHHHHHHAAACERALQYKLGDKIHGFTVNQVTSVPELFLTAVKLTH...,True,
1,6XOT,AAACERALQYKLGDKIHGFTVNQVTSVPELFLTAVKLTHDDTGARY...,968769d61763debeaca404066402f49d,1,966,A,Structural basis for the mechanisms of human p...,10.1038/s41467-022-29322-4,35383169,Nat Commun,...,PMC8983764,RCSB,/content/drive/MyDrive/LLM/Bioreasoner/RCSB_fi...,1,1,1014,d8a0f9bb3577eef783246038c5f4832a,MHHHHHHAAACERALQYKLGDKIHGFTVNQVTSVPELFLTAVKLTH...,True,
2,3ATU,AAAIGIDLGTTYSCVGVFQHGKVEIIANDQGNRTTPSYVAFTDTER...,f69f94b5418e1a09f543250fede4c572,1,379,A,Biochemical and structural studies on the high...,10.1002/pro.663,21608060,Protein Sci.,...,PMC3189522,RCSB,/content/drive/MyDrive/LLM/Bioreasoner/RCSB_fi...,1,1,392,6c2e79e9cf1993d51a68a8e496de1afc,GSFTMAKAAAIGIDLGTTYSCVGVFQHGKVEIIANDQGNRTTPSYV...,True,
3,3AY9,AAAIGIDLGTTYSCVGVFQHGKVEIIANDQGNRTTPSYVAFTDTER...,f69f94b5418e1a09f543250fede4c572,1,379,A,Biochemical and structural studies on the high...,10.1002/pro.663,21608060,Protein Sci.,...,PMC3189522,RCSB,/content/drive/MyDrive/LLM/Bioreasoner/RCSB_fi...,1,1,392,6c2e79e9cf1993d51a68a8e496de1afc,GSFTMAKAAAIGIDLGTTYSCVGVFQHGKVEIIANDQGNRTTPSYV...,True,
4,7E84,AAGVAAWLPFARAAAIGWMPVASGPMPAPPRQERDALIVLNVSGTR...,5f3692cfc07f1d4d4042e7d90fa3a6f4,8,2548,"A,B,D,E,G,H,J,L",Structural basis of gating modulation of Kv4 c...,10.1038/s41586-021-03935-z,34552243,Nature,...,PMC8566240,RCSB,/content/drive/MyDrive/LLM/Bioreasoner/RCSB_fi...,2,2,673,76773e74fa3b1ad8b430b12f3bae588f,AAGVAAWLPFARAAAIGWMPVASGPMPAPPRQERKRTQDALIVLNV...,True,
5,7F3F,AAGVAAWLPFARAAAIGWMPVASGPMPAPPRQERDALIVLNVSGTR...,5f3692cfc07f1d4d4042e7d90fa3a6f4,8,2548,"A,B,D,E,G,H,J,L",Structural basis of gating modulation of Kv4 c...,10.1038/s41586-021-03935-z,34552243,Nature,...,PMC8566240,RCSB,/content/drive/MyDrive/LLM/Bioreasoner/RCSB_fi...,2,2,846,a9737768adeb3a9602d4f6357385a35c,MAAGVAAWLPFARAAAIGWMPVASGPMPAPPRQERKRTQDALIVLN...,True,
6,7ENZ,AAHGSNPEQLKHCNGILKELLSKKHAAYAWPFYKPVDASALGLHDY...,60c2967edcf76b73242618a46369898b,1,115,A,"Structural investigation of a pyrano-1,3-oxazi...",10.1107/S2053230X22001066,35234137,"Acta Crystallogr.,Sect.F",...,PMC8900734,RCSB,/content/drive/MyDrive/LLM/Bioreasoner/RCSB_fi...,1,1,115,60c2967edcf76b73242618a46369898b,AAHGSNPEQLKHCNGILKELLSKKHAAYAWPFYKPVDASALGLHDY...,True,
7,7EO5,AAHGSNPEQLKHCNGILKELLSKKHAAYAWPFYKPVDASALGLHDY...,60c2967edcf76b73242618a46369898b,1,115,A,"Structural investigation of a pyrano-1,3-oxazi...",10.1107/S2053230X22001066,35234137,"Acta Crystallogr.,Sect.F",...,PMC8900734,RCSB,/content/drive/MyDrive/LLM/Bioreasoner/RCSB_fi...,1,1,115,60c2967edcf76b73242618a46369898b,AAHGSNPEQLKHCNGILKELLSKKHAAYAWPFYKPVDASALGLHDY...,True,
8,3I58,AAHIGLRALADLATPMAVRVAATLRVADHIAAGHRTAAEIASAAGA...,65986514710f3c165afef47689a7e961,2,656,"A,B",Molecular basis of substrate promiscuity for t...,10.1021/bi901257q,19702337,Biochemistry,...,PMC4500170,RCSB,/content/drive/MyDrive/LLM/Bioreasoner/RCSB_fi...,1,1,332,e2757cbe34c7e6ef36dafec89dba6cdd,MGKRAAHIGLRALADLATPMAVRVAATLRVADHIAAGHRTAAEIAS...,True,
9,3I64,AAHIGLRALADLATPMAVRVAATLRVADHIAAGHRTAAEIASAAGA...,65986514710f3c165afef47689a7e961,2,656,"A,B",Molecular basis of substrate promiscuity for t...,10.1021/bi901257q,19702337,Biochemistry,...,PMC4500170,RCSB,/content/drive/MyDrive/LLM/Bioreasoner/RCSB_fi...,1,1,332,e2757cbe34c7e6ef36dafec89dba6cdd,MGKRAAHIGLRALADLATPMAVRVAATLRVADHIAAGHRTAAEIAS...,True,



[peek] FASTA-verified duplicate groups (first 20 rows):


,pdb_id,merged_sequence,concat_md5,n_chains,total_length,chain_ids,title,doi,pmid,journal,...,fasta_source,fasta_path,n_fasta_records,n_chains_used,fasta_concat_length,fasta_concat_md5,fasta_concat_sequence,fasta_ok,error,merged_equals_fasta
0,6XOS,AAACERALQYKLGDKIHGFTVNQVTSVPELFLTAVKLTHDDTGARY...,968769d61763debeaca404066402f49d,1,966,A,Structural basis for the mechanisms of human p...,10.1038/s41467-022-29322-4,35383169,Nat Commun,...,RCSB,/content/drive/MyDrive/LLM/Bioreasoner/RCSB_fi...,1,1,1014,d8a0f9bb3577eef783246038c5f4832a,MHHHHHHAAACERALQYKLGDKIHGFTVNQVTSVPELFLTAVKLTH...,True,,False
1,6XOT,AAACERALQYKLGDKIHGFTVNQVTSVPELFLTAVKLTHDDTGARY...,968769d61763debeaca404066402f49d,1,966,A,Structural basis for the mechanisms of human p...,10.1038/s41467-022-29322-4,35383169,Nat Commun,...,RCSB,/content/drive/MyDrive/LLM/Bioreasoner/RCSB_fi...,1,1,1014,d8a0f9bb3577eef783246038c5f4832a,MHHHHHHAAACERALQYKLGDKIHGFTVNQVTSVPELFLTAVKLTH...,True,,False
2,3ATU,AAAIGIDLGTTYSCVGVFQHGKVEIIANDQGNRTTPSYVAFTDTER...,f69f94b5418e1a09f543250fede4c572,1,379,A,Biochemical and structural studies on the high...,10.1002/pro.663,21608060,Protein Sci.,...,RCSB,/content/drive/MyDrive/LLM/Bioreasoner/RCSB_fi...,1,1,392,6c2e79e9cf1993d51a68a8e496de1afc,GSFTMAKAAAIGIDLGTTYSCVGVFQHGKVEIIANDQGNRTTPSYV...,True,,False
3,3AY9,AAAIGIDLGTTYSCVGVFQHGKVEIIANDQGNRTTPSYVAFTDTER...,f69f94b5418e1a09f543250fede4c572,1,379,A,Biochemical and structural studies on the high...,10.1002/pro.663,21608060,Protein Sci.,...,RCSB,/content/drive/MyDrive/LLM/Bioreasoner/RCSB_fi...,1,1,392,6c2e79e9cf1993d51a68a8e496de1afc,GSFTMAKAAAIGIDLGTTYSCVGVFQHGKVEIIANDQGNRTTPSYV...,True,,False
6,7ENZ,AAHGSNPEQLKHCNGILKELLSKKHAAYAWPFYKPVDASALGLHDY...,60c2967edcf76b73242618a46369898b,1,115,A,"Structural investigation of a pyrano-1,3-oxazi...",10.1107/S2053230X22001066,35234137,"Acta Crystallogr.,Sect.F",...,RCSB,/content/drive/MyDrive/LLM/Bioreasoner/RCSB_fi...,1,1,115,60c2967edcf76b73242618a46369898b,AAHGSNPEQLKHCNGILKELLSKKHAAYAWPFYKPVDASALGLHDY...,True,,True
7,7EO5,AAHGSNPEQLKHCNGILKELLSKKHAAYAWPFYKPVDASALGLHDY...,60c2967edcf76b73242618a46369898b,1,115,A,"Structural investigation of a pyrano-1,3-oxazi...",10.1107/S2053230X22001066,35234137,"Acta Crystallogr.,Sect.F",...,RCSB,/content/drive/MyDrive/LLM/Bioreasoner/RCSB_fi...,1,1,115,60c2967edcf76b73242618a46369898b,AAHGSNPEQLKHCNGILKELLSKKHAAYAWPFYKPVDASALGLHDY...,True,,True
8,3I58,AAHIGLRALADLATPMAVRVAATLRVADHIAAGHRTAAEIASAAGA...,65986514710f3c165afef47689a7e961,2,656,"A,B",Molecular basis of substrate promiscuity for t...,10.1021/bi901257q,19702337,Biochemistry,...,RCSB,/content/drive/MyDrive/LLM/Bioreasoner/RCSB_fi...,1,1,332,e2757cbe34c7e6ef36dafec89dba6cdd,MGKRAAHIGLRALADLATPMAVRVAATLRVADHIAAGHRTAAEIAS...,True,,False
9,3I64,AAHIGLRALADLATPMAVRVAATLRVADHIAAGHRTAAEIASAAGA...,65986514710f3c165afef47689a7e961,2,656,"A,B",Molecular basis of substrate promiscuity for t...,10.1021/bi901257q,19702337,Biochemistry,...,RCSB,/content/drive/MyDrive/LLM/Bioreasoner/RCSB_fi...,1,1,332,e2757cbe34c7e6ef36dafec89dba6cdd,MGKRAAHIGLRALADLATPMAVRVAATLRVADHIAAGHRTAAEIAS...,True,,False
10,5K8E,AAIDECLKNAKVPVTARNSTEWKTDASPFNDRLPYTPAAIAKPATV...,32df5728c61893780331dc2c88fdfe94,1,473,A,Discovery of a Xylooligosaccharide Oxidase fro...,10.1074/jbc.M116.741173,27629413,J.Biol.Chem.,...,RCSB,/content/drive/MyDrive/LLM/Bioreasoner/RCSB_fi...,1,1,497,533b7a69d500754a0b7dc59057bc923f,MHLLPLTVSATAVVSAASSPHAKRAAIDECLKNAKVPVTARNSTEW...,True,,False
11,5L6F,AAIDECLKNAKVPVTARNSTEWKTDASPFNDRLPYTPAAIAKPATV...,32df5728c61893780331dc2c88fdfe94,1,473,A,Discovery of a Xylooligosaccharide Oxidase fro...,10.1074/jbc.M116.741173,27629413,J.Biol.Chem.,...,RCSB,/content/drive/MyDrive/LLM/Bioreasoner/RCSB_fi...,1,1,497,533b7a69d500754a0b7dc59057bc923f,MHLLPLTVSATAVVSAASSPHAKRAAIDECLKNAKVPVTARNSTEW...,True,,False


In [ ]:
# === Remove all rows whose pdb_id appears in repeated_proteins_fasta_verified.csv ===
from pathlib import Path
import pandas as pd

IN_VERIFIED = Path(BASE_DIR) / "repeated_proteins_fasta_verified.csv"
IN_ALL      = Path(BASE_DIR) / "3di_chains_chaintag_all.csv"
OUT_FILTER  = Path(BASE_DIR) / "3di_chains_chaintag_all_no_fasta_repeats.csv"
OUT_PDBS    = Path(BASE_DIR) / "fasta_repeat_pdb_ids.csv"   # saved list of removed pdb_ids

assert IN_VERIFIED.exists(), f"Missing: {IN_VERIFIED}"
assert IN_ALL.exists(), f"Missing: {IN_ALL}"

# Load set of PDB IDs to remove
ver = pd.read_csv(IN_VERIFIED, dtype=str)
if "pdb_id" not in ver.columns:
    raise ValueError(f"'pdb_id' column not found in {IN_VERIFIED}")
rem_ids = (
    ver["pdb_id"]
    .dropna()
    .astype(str)
    .str.upper()
    .unique()
    .tolist()
)
rem_set = set(rem_ids)

# Save the list for reference
pd.DataFrame({"pdb_id": sorted(rem_set)}).to_csv(OUT_PDBS, index=False)

print(f"[info] PDB IDs to remove: {len(rem_set)} -> {OUT_PDBS}")

# Stream-filter the big CSV
CHUNKSIZE = 200_000  # adjust if needed
total_in, total_kept, total_removed = 0, 0, 0

# Make sure output doesn't exist yet
if OUT_FILTER.exists():
    OUT_FILTER.unlink()

first_chunk = True
for chunk in pd.read_csv(IN_ALL, dtype=str, chunksize=CHUNKSIZE):
    if "pdb_id" not in chunk.columns:
        raise ValueError(f"'pdb_id' column not found in {IN_ALL}")
    # normalize for robust matching
    chunk["pdb_id"] = chunk["pdb_id"].astype(str).str.upper()

    total_in += len(chunk)
    keep_mask = ~chunk["pdb_id"].isin(rem_set)
    kept = chunk.loc[keep_mask]
    removed = len(chunk) - len(kept)

    total_kept += len(kept)
    total_removed += removed

    kept.to_csv(OUT_FILTER, mode="a", index=False, header=first_chunk)
    first_chunk = False

print("\n=== Removal summary ===")
print(f"Input rows:      {total_in}")
print(f"Removed rows:    {total_removed}")
print(f"Output rows:     {total_kept}")
print(f"Unique pdb_ids removed: {len(rem_set)}")
print(f"[ok] wrote filtered CSV -> {OUT_FILTER}")

[info] PDB IDs to remove: 19655 -> /content/drive/MyDrive/LLM/Bioreasoner/RCSB_files/fasta_repeat_pdb_ids.csv

=== Removal summary ===
Input rows:      418335
Removed rows:    54663
Output rows:     363672
Unique pdb_ids removed: 19655
[ok] wrote filtered CSV -> /content/drive/MyDrive/LLM/Bioreasoner/RCSB_files/3di_chains_chaintag_all_no_fasta_repeats.csv


In [ ]:
# === Count unique pdb_id after removal (streaming) ===
from pathlib import Path
import pandas as pd

OUT_FILTER = Path(BASE_DIR) / "3di_chains_chaintag_all_no_fasta_repeats.csv"
assert OUT_FILTER.exists(), f"Missing filtered CSV: {OUT_FILTER}"

CHUNKSIZE = 200_000  # adjust if needed
remaining_ids = set()

for chunk in pd.read_csv(OUT_FILTER, dtype=str, usecols=["pdb_id"], chunksize=CHUNKSIZE):
    if "pdb_id" not in chunk.columns:
        raise ValueError(f"'pdb_id' column not found in {OUT_FILTER}")
    remaining_ids.update(chunk["pdb_id"].dropna().astype(str).str.upper().unique().tolist())

# Report & save
print(f"[ok] unique pdb_id after removal: {len(remaining_ids)}")

OUT_IDS = Path(BASE_DIR) / "remaining_pdb_ids.csv"
pd.DataFrame({"pdb_id": sorted(remaining_ids)}).to_csv(OUT_IDS, index=False)
print(f"[out] wrote list of remaining pdb_ids -> {OUT_IDS}")

# Quick peek
import itertools
print("Sample remaining IDs:", ", ".join(itertools.islice(iter(sorted(remaining_ids)), 20)))


[ok] unique pdb_id after removal: 91974
[out] wrote list of remaining pdb_ids -> /content/drive/MyDrive/LLM/Bioreasoner/RCSB_files/remaining_pdb_ids.csv
Sample remaining IDs: 100D, 109D, 112D, 114D, 117D, 11BA, 126D, 129L, 130L, 131L, 133D, 149L, 150L, 151L, 152L, 165D, 16VP, 183D, 1914, 191D
